# Aula 01 — Great Expectations: Suites e Checkpoints

Neste notebook, exploramos o conceito de **expectativas declarativas** para
validacao de dados em pipelines de ML.

## Conceitos-chave

| Conceito | Descricao |
|----------|----------|
| **Expectation** | Regra declarativa sobre uma coluna (ex: "age deve estar entre 0 e 95") |
| **Suite** | Conjunto nomeado de expectativas que formam um contrato de qualidade |
| **Checkpoint** | Execucao de uma suite sobre um dataset, gerando resultado consolidado |
| **Fallback local** | Mesma semantica implementada sem depender do Great Expectations |

## Por que usar expectativas declarativas?

Em vez de escrever codigo imperativo como `assert df['age'].min() >= 0`,
declaramos regras como especificacoes. Isso separa **o que validar** de
**como executar**, permitindo:

- Reutilizar suites em diferentes ambientes
- Documentar contratos de dados automaticamente
- Trocar o backend de execucao sem mudar as regras

In [ ]:
import logging
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

## 1. Definindo expectativas como especificacoes

Cada expectativa e uma regra **declarativa**: define o que se espera de uma
coluna sem dizer como verificar. O motor de avaliacao cuida da execucao.

In [ ]:
from ge_validation import (
    ExpectationKind,
    ExpectationSpec,
    build_churn_suite,
    build_basic_titanic_suite,
)

# Visualizar a suite de churn
churn_suite = build_churn_suite()
print(f"Suite de churn: {len(churn_suite)} expectativas\n")
for spec in churn_suite:
    print(f"  {spec.kind.value:8s} | {spec.column:20s} | {spec.name}")
    if spec.description:
        print(f"           {spec.description}")

## 2. Gerando datasets sinteticos

Criamos dois datasets: um **valido** (que deve passar em todas as
expectativas) e um **invalido** (com erros propositais).

In [ ]:
from ge_validation import build_churn_examples

valid_df, invalid_df = build_churn_examples()

print(f"Dataset valido: {valid_df.shape}")
print(f"Dataset invalido: {invalid_df.shape}")
print("\nPrimeiras linhas do dataset invalido:")
invalid_df.head(12)

## 3. Executando o checkpoint

O checkpoint aplica **todas** as expectativas da suite sobre o dataset
e consolida em um resultado unico: aprovado ou reprovado.

In [ ]:
from ge_validation import run_checkpoint, format_checkpoint_report

# Checkpoint no dataset valido
result_valid = run_checkpoint("churn_valid", valid_df, churn_suite)
print(format_checkpoint_report(result_valid))

In [ ]:
# Checkpoint no dataset invalido
result_invalid = run_checkpoint("churn_invalid", invalid_df, churn_suite)
print(format_checkpoint_report(result_invalid))

## 4. Exportando resultados como JSON

Em producao, os resultados do checkpoint sao exportados para sistemas
de monitoramento, dashboards ou CI/CD.

In [ ]:
import json
from ge_validation import export_checkpoint_json

report_json = export_checkpoint_json(result_invalid)
print(json.dumps(report_json, indent=2, ensure_ascii=False))

## 5. Criando sua propria suite

Exercicio: crie uma suite personalizada para um dataset de sua escolha.
Use os tipos de expectativa disponiveis:
- `MIN` / `MAX`: valor minimo/maximo
- `BETWEEN`: faixa de valores
- `SET`: valores permitidos
- `MISSING`: taxa maxima de nulos
- `UNIQUE`: sem duplicatas
- `NOT_NULL`: nenhum nulo

In [ ]:
from ge_validation import evaluate_spec

# Exemplo: suite customizada para dados de vendas
custom_suite = (
    ExpectationSpec("preco_positivo", "preco", ExpectationKind.MIN, 0.01,
                    description="Preco deve ser positivo"),
    ExpectationSpec("qtd_range", "quantidade", ExpectationKind.BETWEEN, (1, 1000),
                    description="Quantidade entre 1 e 1000"),
    ExpectationSpec("categoria_valida", "categoria", ExpectationKind.SET,
                    ("eletronico", "vestuario", "alimento"),
                    description="Apenas categorias cadastradas"),
)

# Testar com dados sinteticos
vendas_df = pd.DataFrame({
    "preco": [29.90, 149.90, 5.50, -1.0],
    "quantidade": [2, 1, 50, 1500],
    "categoria": ["eletronico", "vestuario", "alimento", "brinquedo"],
})

for spec in custom_suite:
    result = evaluate_spec(vendas_df, spec)
    status = "PASS" if result.success else "FAIL"
    print(f"[{status}] {result.name}: {result.observed_value}")

## Resumo

- Expectativas declarativas separam **regras** de **execucao**.
- Uma **suite** agrupa expectativas em um contrato testavel.
- Um **checkpoint** executa a suite e consolida o veredicto.
- O resultado pode ser exportado como JSON para integracao com CI/CD.
- O fallback local garante que o material funciona sem instalar GX.